# Évaluation des tâches NLP : exemples et métriques
Master Data & IA – INPT, *ML Evaluation* 2026

Une cellule par tâche : **1. Classification**, **2. Sequence labeling (NER)**, **3. Traduction automatique**.
Exécuter d'abord la cellule d'installation (Google Colab recommandé).

In [1]:
!pip install -q scikit-learn seqeval sacrebleu transformers sentencepiece torch

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Classification : analyse de sentiment
On entraîne un modèle TF-IDF + régression logistique sur un petit jeu de phrases (1 = positif, 0 = négatif), puis on mesure ses performances sur un jeu de test.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, roc_auc_score)

import random
random.seed(0)
# Jeu de données synthétique : phrases construites à partir de gabarits (vocabulaire partagé entre classes)
subjects = ["movie", "film", "story", "acting", "soundtrack", "ending", "plot", "cast"]
pos_words = ["great", "wonderful", "brilliant", "amazing", "excellent", "superb", "delightful", "touching"]
neg_words = ["terrible", "boring", "awful", "disappointing", "horrible", "weak", "silly", "messy"]
frames = ["The {s} was {w}", "I found the {s} really {w}", "What a {w} {s}", "Honestly, the {s} is {w}"]

texts, labels = [], []
for _ in range(300):
    label = random.randint(0, 1)
    w = random.choice(pos_words if label else neg_words)
    texts.append(random.choice(frames).format(s=random.choice(subjects), w=w))
    # 10 % de bruit d'annotation pour que les métriques ne soient pas parfaites
    labels.append(1 - label if random.random() < 0.10 else label)

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.3,
                                                    stratify=labels, random_state=42)
model = make_pipeline(TfidfVectorizer(), LogisticRegression())
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average="binary")
print(f"Precision={p:.3f}  Recall={r:.3f}  F1={f:.3f}")
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob), 3))
print("\nMatrice de confusion (lignes = vrai, colonnes = prédit) :\n", confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred, target_names=["négatif", "positif"]))

Accuracy : 0.856
Precision=0.911  Recall=0.820  F1=0.863
ROC-AUC  : 0.848

Matrice de confusion (lignes = vrai, colonnes = prédit) :
 [[36  4]
 [ 9 41]]

               precision    recall  f1-score   support

     négatif       0.80      0.90      0.85        40
     positif       0.91      0.82      0.86        50

    accuracy                           0.86        90
   macro avg       0.86      0.86      0.86        90
weighted avg       0.86      0.86      0.86        90



## 2. Sequence labeling : reconnaissance d'entités nommées (NER)
Un modèle BERT pré-entraîné prédit les entités (PER, ORG, LOC). On convertit sa sortie au format BIO et on la compare à une annotation manuelle avec **seqeval** (évaluation **par entité complète**).

In [3]:
from transformers import pipeline
from seqeval.metrics import classification_report as seq_report, f1_score, precision_score, recall_score
from seqeval.metrics import accuracy_score as token_accuracy

ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

# Phrases annotées à la main (schéma BIO) : (phrase, étiquettes de référence)
data = [
    ("Mohammed VI visited Casablanca last week",       ["B-PER","I-PER","O","B-LOC","O","O"]),
    ("Omar works at Microsoft in Paris",               ["B-PER","O","O","B-ORG","O","B-LOC"]),
    ("Google opened a new office in Rabat",            ["B-ORG","O","O","O","O","O","B-LOC"]),
    ("Emmanuel Macron met Angela Merkel in Berlin",    ["B-PER","I-PER","O","B-PER","I-PER","O","B-LOC"]),
    ("The University of Oxford is in England",         ["O","B-ORG","I-ORG","I-ORG","O","O","B-LOC"]),
]

def to_bio(sentence):
    """Prédit les entités puis les convertit en étiquettes BIO alignées sur les mots."""
    ents = ner(sentence)
    tags, pos = [], 0
    for word in sentence.split():
        start = sentence.index(word, pos); end = start + len(word); pos = end
        tag = "O"
        for e in ents:
            if e["start"] <= start and end <= e["end"]:
                tag = ("B-" if start == e["start"] else "I-") + e["entity_group"]
        tags.append(tag)
    return tags

y_true = [gold for _, gold in data]
y_pred = [to_bio(s) for s, _ in data]

for (s, gold), pred in zip(data, y_pred):
    print(s, "\n  gold:", gold, "\n  pred:", pred)

print("\nAccuracy token (gonflée par 'O') :", round(token_accuracy(y_true, y_pred), 3))
print("Precision entité :", round(precision_score(y_true, y_pred), 3))
print("Recall entité    :", round(recall_score(y_true, y_pred), 3))
print("F1 entité        :", round(f1_score(y_true, y_pred), 3))
print("\n", seq_report(y_true, y_pred))

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Mohammed VI visited Casablanca last week 
  gold: ['B-PER', 'I-PER', 'O', 'B-LOC', 'O', 'O'] 
  pred: ['B-PER', 'I-PER', 'O', 'B-LOC', 'O', 'O']
Omar works at Microsoft in Paris 
  gold: ['B-PER', 'O', 'O', 'B-ORG', 'O', 'B-LOC'] 
  pred: ['B-PER', 'O', 'O', 'B-ORG', 'O', 'B-LOC']
Google opened a new office in Rabat 
  gold: ['B-ORG', 'O', 'O', 'O', 'O', 'O', 'B-LOC'] 
  pred: ['B-ORG', 'O', 'O', 'O', 'O', 'O', 'O']
Emmanuel Macron met Angela Merkel in Berlin 
  gold: ['B-PER', 'I-PER', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC'] 
  pred: ['B-PER', 'I-PER', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC']
The University of Oxford is in England 
  gold: ['O', 'B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'B-LOC'] 
  pred: ['O', 'B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'B-LOC']

Accuracy token (gonflée par 'O') : 0.97
Precision entité : 1.0
Recall entité    : 0.917
F1 entité        : 0.957

               precision    recall  f1-score   support

         LOC       1.00      0.80      0.89         5
         ORG       1

## 3. Traduction automatique : EN → FR
Un modèle Marian (Helsinki-NLP) traduit des phrases anglaises. On compare avec des traductions de référence via **BLEU**, **chrF** et **TER** (sacrebleu).

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def translate(sentences):
    batch = tokenizer(sentences, return_tensors="pt", padding=True)
    outputs = mt_model.generate(**batch, num_beams=4, max_new_tokens=128)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

sources = ["The cat is sitting on the mat.",
           "I would like to book a flight to Rabat.",
           "The students are preparing their homework.",
           "Machine learning models need a lot of data.",
           "The weather is beautiful today."]
references = ["Le chat est assis sur le tapis.",
              "Je voudrais réserver un vol pour Rabat.",
              "Les étudiants préparent leurs devoirs.",
              "Les modèles d'apprentissage automatique ont besoin de beaucoup de données.",
              "Il fait beau aujourd'hui."]

hyps = translate(sources)

for s, h, r in zip(sources, hyps, references):
    print(f"SRC : {s}\nHYP : {h}\nREF : {r}\n  BLEU phrase = {sacrebleu.sentence_bleu(h, [r]).score:.1f}\n")

refs = [references]  # sacrebleu attend une liste de listes de références
print("BLEU corpus :", round(sacrebleu.corpus_bleu(hyps, refs).score, 2))
print("chrF corpus :", round(sacrebleu.corpus_chrf(hyps, refs).score, 2))
print("TER  corpus :", round(sacrebleu.corpus_ter(hyps, refs).score, 2), "(plus bas = mieux)")

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


... (4 outputs left out)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SRC : The cat is sitting on the mat.
HYP : Le chat est assis sur le tapis.
REF : Le chat est assis sur le tapis.
  BLEU phrase = 100.0

SRC : I would like to book a flight to Rabat.
HYP : Je voudrais réserver un vol pour Rabat.
REF : Je voudrais réserver un vol pour Rabat.
  BLEU phrase = 100.0

SRC : The students are preparing their homework.
HYP : Les élèves préparent leurs devoirs.
REF : Les étudiants préparent leurs devoirs.
  BLEU phrase = 53.7

SRC : Machine learning models need a lot of data.
HYP : Les modèles d'apprentissage automatique ont besoin de beaucoup de données.
REF : Les modèles d'apprentissage automatique ont besoin de beaucoup de données.
  BLEU phrase = 100.0

SRC : The weather is beautiful today.
HYP : Le temps est beau aujourd'hui.
REF : Il fait beau aujourd'hui.
  BLEU phrase = 30.2

BLEU corpus : 84.15
chrF corpus : 91.21
TER  corpus : 12.12 (plus bas = mieux)


## Conclusion
| Tâche | Métrique principale | Pourquoi |
|---|---|---|
| Classification | F1 (+ AUC) | robuste au déséquilibre des classes |
| NER | F1 par entité | exige type **et** frontières exacts |
| Traduction | chrF / BLEU (+ COMET) | mesure la ressemblance avec une référence |